# Semantic Product Search Engine — Kaggle Quickstart

Run this on **Kaggle with GPU + Internet enabled** (Session options, right panel).
It clones the branch, installs dependencies, and runs the tagging pipeline on a
few sample products — no Supabase needed yet.

See `docs/PIPELINE.md` for the full design.

## 1. Clone the branch (skips if already present)

In [ ]:
import os

REPO = "https://github.com/MaximizeAy/Semantic-Search-Engine.git"
BRANCH = "claude/getting-started-2s6q6h"
DIR = "/kaggle/working/Semantic-Search-Engine"

if not os.path.isdir(DIR):
    !git clone --branch {BRANCH} --single-branch {REPO} {DIR}
else:
    !cd {DIR} && git pull origin {BRANCH}

import sys
sys.path.insert(0, DIR)
os.chdir(DIR)
print("repo ready:", DIR)

## 2. Install dependencies

First run pulls the `bge-m3` weights (~2.2 GB) when the model first loads.

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
print("deps installed")

## 3. Build the pipeline

Defaults to the seed taxonomy (21 Google Product Taxonomy categories + starter tag vocabulary).

In [ ]:
from semantic_search.pipeline import build_tag_pipeline
from semantic_search import taxonomy

print(f"{len(taxonomy.CATEGORIES)} categories, {len(taxonomy.STARTER_TAGS)} starter tags")

pipeline = build_tag_pipeline()  # uses the seed taxonomy by default
print("pipeline stages:", [s.name for s in pipeline.stages])

## 4. Tag some sample products

In [ ]:
samples = [
    ("Wireless Bluetooth Headphones", "Over-ear noise cancelling headphones with 30h battery and fast charging."),
    ("Stainless Steel Water Bottle", "Insulated reusable bottle, keeps drinks cold 24h. BPA-free."),
    ("Organic Cotton T-Shirt", "Breathable unisex tee made from 100% organic cotton."),
    ("4K Gaming Monitor 27 inch", "144Hz HD display with adjustable stand and USB-C."),
    ("Ceramic Non-Stick Frying Pan", "Dishwasher safe cookware with ergonomic handle."),
]

for name, desc in samples:
    rec = pipeline.run_text(name, desc)
    print(f"\n{name}")
    print(f"  category : {rec['category']}")
    print(f"  tags     : {rec['tags']}")
    if rec['discovered_tags']:
        print(f"  discovered: {rec['discovered_tags']}")

## 5. Typo-tolerant check

The fuzzy stage canonicalizes noisy terms against the vocabulary.

In [ ]:
from semantic_search.fuzzy import FuzzyCanonicalizer
from semantic_search import taxonomy

canon = FuzzyCanonicalizer(taxonomy.build_vocabulary())
for noisy in ["wireles", "blutooth", "stainles steel", "headphnes"]:
    print(f"{noisy!r:18} -> {canon.correct_query(noisy)!r}")

## Next steps

- Create a dedicated **Supabase** project, enable `pgvector`, and run `db/schema.sql`.
- Add `SUPABASE_URL` / `SUPABASE_KEY` via **Add-ons → Secrets**.
- Use `scripts/index_catalog.py` (coming) to index a product CSV into Supabase,
  then `build_search_pipeline(store)` for typo-tolerant semantic search.